In [ ]:
# Allows you to use modified modules without rebooting the kernel
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller

df = pd.read_json("small_dBx.json")
display(df.head())

In [ ]:
def grangers_causation_matrix(
    data, variables, test="ssr_chi2test", verbose=False, maxlag=10
):
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table
    are the P-Values. P-Values lesser than the significance level (0.05), implies
    the Null Hypothesis that the coefficients of the corresponding past values is
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.
    """
    df = pd.DataFrame(
        np.zeros((len(variables), len(variables))), columns=variables, index=variables
    )
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(
                data[[r, c]], maxlag=maxlag, verbose=False
            )
            p_values = [round(test_result[i + 1][0][test][1], 4) for i in range(maxlag)]
            if verbose:
                print(f"Y = {r}, X = {c}, P Values = {p_values}")
            min_p_value = np.min(p_values)
            df.loc[r, c] = min_p_value
    df.columns = [var + "_x" for var in variables]
    df.index = [var + "_y" for var in variables]
    return df


GCM = grangers_causation_matrix(df, variables=df.columns)
GCM.to_excel("grangers_causation_matrix.xlsx")

In [ ]:
try:
    df.drop(columns=[f"t_{i}" for i in range(20)], inplace=True)
except KeyError:
    pass
finally:
    display(df.head())

In [ ]:
nobs = 40
df_train, df_test = df[0:-nobs], df[-nobs:]

# Check size
print(df_train.shape)  # (119, 8)
print(df_test.shape)  # (4, 8)

In [ ]:
def adfuller_test(series, signif=0.05, name="", verbose=False, nostatonly=False):
    """Perform ADFuller to test for Stationarity of given series and print report"""
    r = adfuller(series, autolag="AIC")
    output = {
        "test_statistic": round(r[0], 4),
        "pvalue": round(r[1], 4),
        "n_lags": round(r[2], 4),
        "n_obs": r[3],
    }
    p_value = output["pvalue"]

    def adjust(val, length=6):
        return str(val).ljust(length)

    print(f'    Augmented Dickey-Fuller Test on "{name}"', "\n   ", "-" * 47)
    if verbose:
        # Print Summary
        print(" Null Hypothesis: Data has unit root. Non-Stationary.")
        print(f" Significance Level    = {signif}")
        print(f' Test Statistic        = {output["test_statistic"]}')
        print(f' No. Lags Chosen       = {output["n_lags"]}')
        for key, val in r[4].items():
            print(f" Critical value {adjust(key)} = {round(val, 3)}")

    if p_value <= signif:
        if not nostatonly:
            print(f" => P-Value = {p_value}. Rejecting Null Hypothesis.")
            print(" => Series is Stationary.")
    else:
        print(f" => P-Value = {p_value}. Weak evidence to reject the Null Hypothesis.")
        print(" => Series is Non-Stationary.")

In [ ]:
# ADF Test on each column
for name, column in df_train.items():
    adfuller_test(column, name=column.name, nostatonly=True)
    print("\n")

In [ ]:
# 1st difference
df_differenced = df_train.diff().dropna()
# ADF Test on each column of 1st Differences Dataframe
for name, column in df_differenced.items():
    adfuller_test(column, name=column.name, nostatonly=True)
    print("\n")

In [ ]:
model = VAR(df_differenced)
for i in [1, 2, 3, 4, 5, 6, 7, 8, 9]:
    result = model.fit(i)
    print("Lag Order =", i)
    print("AIC : ", result.aic)
    print("BIC : ", result.bic)
    print("FPE : ", result.fpe)
    print("HQIC: ", result.hqic, "\n")

## Apply different forecast methods

In [ ]:
window = (4200, 1)

# 3.0. Pre-process dataset

https://mlpills.dev/time-series/step-by-step-guide-to-multivariate-time-series-forecasting-with-var-models/

In [ ]:
# df_roll = extended_dataset.rolling(60).mean().dropna()
df_roll = extended_Bx_dataset

# Split into train and test
cutoff_index = int(df_roll.shape[0] * 0.9)
df_train = df_roll.iloc[:cutoff_index]
df_test = df_roll.iloc[cutoff_index:]

# Apply differencing
df_diff = df_train.diff().dropna()

In [ ]:
# # Check if data is stationary
# from statsmodels.tsa.stattools import adfuller

# for variable in df_diff.columns:

#     # Perform the ADF test
#     result = adfuller(df_diff[variable])

#     # Extract and print the p-value from the test result
#     p_value = result[1]
#     print("p-value:", p_value)

#     # Interpret the result
#     if p_value <= 0.05:
#         print(f"The variable {variable} is stationary.\n")
#     else:
#         print(f"The variable {variable} is not stationary.\n")

In [ ]:
# display(df_diff.corr())

In [ ]:
# Import StandardScaler
from sklearn.preprocessing import StandardScaler

# Instantiate the scaler
scaler = StandardScaler()

# Transform data
scaled_values = scaler.fit_transform(df_diff)

# Convert to dataframe
df_scaled = pd.DataFrame(scaled_values, columns=df_diff.columns, index=df_diff.index)

In [ ]:
# Define function for data transformation
def df_test_transformation(df, test_start_date, scaler):
    # Apply differencing to make data stationary
    df_diff = df.diff().dropna()

    # Scale data using the previously defined scaler
    df_scaled = pd.DataFrame(
        scaler.fit_transform(df_diff), columns=df_diff.columns, index=df_diff.index
    )

    # Select only the data that belongs to the testing set
    df_test_processed = df_scaled[df_scaled.index > test_start_date]

    return df_test_processed


# Define function for inverting data transformation
def df_inv_transformation(df_processed, df, scaler):
    # Invert StandardScaler transformation
    df_diff = pd.DataFrame(
        scaler.inverse_transform(df_processed),
        columns=df_processed.columns,
        index=df_processed.index,
    )

    # Invert differenting
    df_original = df_diff.cumsum() + df[df.index < df_diff.index[0]].iloc[-1]

    return df_original


# Apply function to our data
df_test_processed = df_test_transformation(df_roll, df_scaled.index[-1], scaler)

In [ ]:
# Import library
from statsmodels.tsa.vector_ar.var_model import VAR

# Insantiate VAR model
model = VAR(df_scaled)

In [ ]:
# Get optimal lag order based on the four criteria
optimal_lags = model.select_order()

print(f"The optimal lag order selected: {optimal_lags.selected_orders}")

In [ ]:
# Fit the model after selecting the lag order
# lag_order = optimal_lags.selected_orders['aic']
lag_order = 12
results = model.fit(lag_order)

# # Estimate the model (VAR) and show summary
# var_model = results.model
# a = results.summary()
# a

In [ ]:
# Forecast next five minutes
horizon = 10
forecast = results.forecast(df_scaled.values[-lag_order:], steps=horizon)

# Convert to dataframe
df_forecast = pd.DataFrame(
    forecast, columns=df_scaled.columns, index=df_test.iloc[:horizon].index
)

In [ ]:
import matplotlib.pyplot as plt

# Plot forecasted increment of cases
comp = "Bx"
ax = df_scaled[-30:][comp].plot(figsize=(10, 5))
df_test_processed[:horizon][comp].plot(ax=ax)
df_forecast[comp].plot(ax=ax)
plt.grid(alpha=0.5, which="both")
plt.xlabel("Date")
plt.ylabel("Increment " + comp)
plt.legend(["Train", "Test", "Forecast"])
plt.show()

# 3. Apply VAR model on Bx series with different replenishment.

First of all we need to split original dataset on testing and validation datasets. 